# Next Version 

In [4]:
import dfBasics
import pandas as pd

version_sla = 'v00001'
version     = version_sla + '/v00000'

home_directory  =  '/home/jovyan/work/'
share_directory =  '/home/jovyan/work/share/'
#share_directory =  '/home/jovyan/share/'

In [5]:
sparkSession = dfBasics.getSparkSession()

In [8]:
!ls /home/jovyan/work/share/

data  demo  sla


In [6]:
df = sparkSession.read.parquet(share_directory + 'data/sla_enc_all_v00001_v00001.parquet')
senders = df.select('CSENDERENDPOINTID').dropDuplicates().toPandas()['CSENDERENDPOINTID']
len(senders)
df.show(n=5)
#df.dtypes

AnalysisException: Path does not exist: file:/home/jovyan/work/share/data/sla_enc_all_v00001_v00001.parquet

In [12]:
import pyspark.sql.functions as f
sender = int(senders[0])
pfall = df.where(f.col("CSENDERENDPOINTID").isin([sender])).toPandas()

In [16]:
pd.unique(pfall['CSTATUS'])

array([14, 11,  2, 10], dtype=int32)

In [17]:
df3 = df.where(f.col("CGLOBALMESSAGEID").isin(['2a3df189-699c-11ec-90fd-4ad4ac1e100f']))

In [18]:
df3.show()

+--------------------+-------------+-------------+-------+--------+-----------------+---------------+------------+-----------------+-------------------+-------+------------+----------------+----+-----+---+----+------+
|    CGLOBALMESSAGEID|   CSTARTTIME|     CENDTIME|CSTATUS|CSERVICE|CSENDERENDPOINTID|CSENDERPROTOCOL|CINBOUNDSIZE|CRECEIVERPROTOCOL|CRECEIVERENDPOINTID|CSLATAT|CMESSAGETAT2|CSLADELIVERYTIME|year|month|day|hour|minute|
+--------------------+-------------+-------------+-------+--------+-----------------+---------------+------------+-----------------+-------------------+-------+------------+----------------+----+-----+---+----+------+
|2a3df189-699c-11e...|1640887999475|1640888000287|     11|       5|             4946|              0|         511|               -1|                 -1|      0|         812|              -1|2021|   12| 30|  19|    13|
+--------------------+-------------+-------------+-------+--------+-----------------+---------------+------------+--------------

In [24]:
pfall = df3.toPandas()

In [22]:
inverse_transform([11],column='CSTATUS')

['PENDING']

In [29]:
assign_outcome_error(pfall)

,CGLOBALMESSAGEID,CSTARTTIME,CENDTIME,CSTATUS,CSERVICE,CSENDERENDPOINTID,CSENDERPROTOCOL,CINBOUNDSIZE,CRECEIVERPROTOCOL,CRECEIVERENDPOINTID,CSLATAT,CMESSAGETAT2,CSLADELIVERYTIME,year,month,day,hour,minute,error
0,2a3df189-699c-11ec-90fd-4ad4ac1e100f,1640887999475,1640888000287,11,5,4946,0,511,-1,-1,0,812,-1,2021,12,30,19,13,1


In [31]:
columns = ['CGLOBALMESSAGEID',  'CSTARTTIME', 'CENDTIME', 'CSTATUS', 'CSERVICE', 'CSENDERENDPOINTID', 'CSENDERPROTOCOL', 'CINBOUNDSIZE', 'CRECEIVERPROTOCOL', 'CRECEIVERENDPOINTID', 'CSLATAT', 'CMESSAGETAT2', 'CSLADELIVERYTIME']
df_org  = sparkSession.read.parquet('hdfs://172.30.17.145:8020/sla_sql_data/*/*').select(columns).dropDuplicates() 
df_org3 =  df_org.where(f.col("CGLOBALMESSAGEID").isin(['2a3df189-699c-11ec-90fd-4ad4ac1e100f']))
pfall_org_3 = df_org3.toPandas()

In [32]:
pfall_org_3

,CGLOBALMESSAGEID,CSTARTTIME,CENDTIME,CSTATUS,CSERVICE,CSENDERENDPOINTID,CSENDERPROTOCOL,CINBOUNDSIZE,CRECEIVERPROTOCOL,CRECEIVERENDPOINTID,CSLATAT,CMESSAGETAT2,CSLADELIVERYTIME
0,2a3df189-699c-11ec-90fd-4ad4ac1e100f,1640887999475,1640888000287,PENDING,MAKO_BE_ROUTER,e57c94c0-e0f0-11e8-be62-528eac1b495c,AS2,511,None,None,0,812,-1


In [33]:
df_status_org = sparkSession.read.parquet('hdfs://172.30.17.145:8020/sla_sql_data/*/*').select(['CSTATUS']).dropDuplicates().toPandas() 

In [35]:
df_status = df.select(['CSTATUS']).dropDuplicates().toPandas() 

In [37]:
df_status_org
df_status

transform('PENDING',column='CSTATUS')

11

In [40]:
# COMPONENT_ERROR 0
for status in set(df_status_org['CSTATUS']):
    print(status,transform(status,column='CSTATUS'))

COMPONENT_ERROR_CONTENTEXTRACTOR 1
PENDING 11
CONFIGURATION_ERROR 2
EXPIRED 7
DATA_ERROR 3
SUCCESS_POLLQUEUE 16
TORESEND 18
RECALLED 13
MESSAGE_QUEUED 10
DELETED 4
WARNING 19
COMPONENT_ERROR 0
THREAT_DETECTED 17
MDN_ERROR 9
ECHO-ERROR 5
FAILED 8
ERROR 6
READY_FOR_DOWNLOAD 12
SUCCESS 14
SUCCESS_DOWNLOADED 15


# Main

In [9]:
import dfBasics
import pandas as pd

version_sla = 'v00001'
version     = version_sla + '/v00000'

home_directory  =  '/home/jovyan/work/'
share_directory =  '/home/jovyan/work/share/'
#share_directory =  '/home/jovyan/share/'

In [10]:
sparkSession = dfBasics.getSparkSession()

In [ ]:
#senders = list(sparkSession.read.parquet('hdfs://172.30.17.145:8020/user/admin/sla/v00000/v00000/senders/senders.parquet').toPandas()['CSENDERENDPOINTID'])

In [ ]:
#senders

In [11]:
senders_files = sparkSession.read.options(delimiter=',') \
                      .csv('hdfs://172.30.17.145:8020/user/admin/sla/' + version + '/senders/senders_files.txt').toPandas()
senders_files.columns = ['size','filename']    

In [12]:
filename = senders_files['filename'][0]
failed = []
for filename in senders_files['filename']:
    try:
        sparkSession.read.text('hdfs://172.30.17.145:8020/user/admin/sla/' + version + '/encoded/senders/' + filename + '/_SUCCESS')
    except Exception as exception: 
        if not 'temporary' in filename:
            failed.append(filename.split('sla_enc_')[1].split('.')[0])

In [13]:
senders_files

,size,filename
0,20,sla_enc_000b4e02-7f43-11ec-ae40-78d9ac1e124d.p...
1,16,sla_enc_00171070-9207-11eb-9829-4c38ac1e124c.p...
2,44,sla_enc_00323780-e749-11e8-be62-528eac1b495c.p...
3,36,sla_enc_004f4c00-f237-11e8-a4f3-b62cac1b495c.p...
4,16,sla_enc_0055c970-1dcf-11eb-b43d-596fac1b495c.p...
...,...,...
3460,16,sla_enc_ff987e60-a16a-11e9-a189-ddafac1b495c.p...
3461,16,sla_enc_ffdcb870-87f3-11eb-b182-a224ac1e124c.p...
3462,292,sla_enc_ffe5afd0-ecc7-11e8-a40c-5396ac1b495c.p...
3463,16,sla_enc_fff8bb00-a213-11e9-a189-ddafac1b495c.p...


# Examples

## some info

In [ ]:
len(senders_files),max(senders_files['size'])

## get message tracking data 

### single file

In [14]:
def get_empty_message_tracking_dataframe():
    columns = ['CGLOBALMESSAGEID', 'CSTARTTIME', 'CENDTIME', 'CSTATUS', 'CSERVICE', \
       'CSENDERENDPOINTID', 'CSENDERPROTOCOL', 'CINBOUNDSIZE', \
       'CRECEIVERPROTOCOL', 'CRECEIVERENDPOINTID', 'CSLATAT', 'CMESSAGETAT2', \
       'CSLADELIVERYTIME', 'year', 'month', 'day', 'hour', 'minute']
    return pd.DataFrame(columns=columns)

def get_message_tracking_dataframe_of_senders_filename(filename, version=version):
    try:
        return sparkSession.read.parquet('hdfs://172.30.17.145:8020/user/admin/sla/' + version + '/encoded/senders/' + filename).toPandas()
    except Exception as exception: 
        return get_empty_message_tracking_dataframe()

In [15]:
pfall = get_message_tracking_dataframe_of_senders_filename(senders_files['filename'][1])    

In [16]:
pfall.head()

,CGLOBALMESSAGEID,CSTARTTIME,CENDTIME,CSTATUS,CSERVICE,CSENDERENDPOINTID,CSENDERPROTOCOL,CINBOUNDSIZE,CRECEIVERPROTOCOL,CRECEIVERENDPOINTID,CSLATAT,CMESSAGETAT2,CSLADELIVERYTIME,year,month,day,hour,minute
0,175cf1c1-4afa-11ed-be5c-0905ac1e100e,1665667351259,1665667357772,14,6,2,4,33672,0,4870,6466,6513,1665667357725,2022,10,13,15,22
1,ee445b1f-4afa-11ed-b5bc-15f0ac1e100d,1665667711809,1665667716813,14,6,2,4,33664,0,3932,4957,5004,1665667716766,2022,10,13,15,28
2,42d298e3-4ad9-11ed-b5bc-15f0ac1e100d,1665653250781,1665653255722,14,6,2,4,33820,0,2382,4909,4941,1665653255690,2022,10,13,11,27
3,6ca7539c-4ad8-11ed-be5c-0905ac1e100e,1665652891465,1665652897927,14,6,2,4,33659,0,3476,6415,6462,1665652897880,2022,10,13,11,21
4,6b5bcb58-4ad9-11ed-be5c-0905ac1e100e,1665653318789,1665653325777,14,6,2,4,33652,0,4102,6957,6988,1665653325746,2022,10,13,11,28


In [17]:
len(pd.unique(pfall['CGLOBALMESSAGEID'])),len(pfall), pd.unique(pfall['CSTATUS']), pd.unique(pfall['CSERVICE'])

(13, 13, array([14], dtype=int32), array([6], dtype=int32))

In [18]:
pfall[ pfall['CSERVICE'] == -1]

,CGLOBALMESSAGEID,CSTARTTIME,CENDTIME,CSTATUS,CSERVICE,CSENDERENDPOINTID,CSENDERPROTOCOL,CINBOUNDSIZE,CRECEIVERPROTOCOL,CRECEIVERENDPOINTID,CSLATAT,CMESSAGETAT2,CSLADELIVERYTIME,year,month,day,hour,minute


### create parquet file with all senders

In [19]:
df = sparkSession.read.parquet('hdfs://172.30.17.145:8020/user/admin/sla/' + version + '/encoded/senders/*')
df.write.mode("overwrite").parquet("/home/jovyan/work/output/sla_enc_all" + ".parquet")
#pfall = df.toPandas()

Py4JJavaError: An error occurred while calling o17440.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 5.0 failed 1 times, most recent failure: Lost task 0.0 in stage 5.0 (TID 3471) (94757c6facaa executor driver): org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:301)
	at org.apache.spark.util.ThreadUtils$.parmap(ThreadUtils.scala:375)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.readParquetFootersInParallel(ParquetFileFormat.scala:481)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1(ParquetFileFormat.scala:527)
	at org.apache.spark.sql.execution.datasources.parquet.ParquetFileFormat$.$anonfun$mergeSchemasInParallel$1$adapted(ParquetFileFormat.scala:521)
	at org.apache.spark.sql.execution.datasources.SchemaMergeUtils$.$anonfun$mergeSchemasInParallel$2(SchemaMergeUtils.s

### all senders

In [ ]:
df = sparkSession.read.parquet('hdfs://172.30.17.145:8020/user/admin/sla/' + version + '/encoded/sla_enc_all' + ".parquet")

In [ ]:
pfall = df.toPandas()

## check message tracking files

In [ ]:
def get_skip_senders(senders_files):
    skip_senders_files = []
    for file in senders_files:
        try:
            pfall = sparkSession.read.parquet('hdfs://172.30.17.145:8020/user/admin/sla/' + version + '/encoded/senders/' + file).toPandas()
            if -1 in pd.unique(pfall['CSENDERENDPOINTID']) :
                skip_senders_files.append(file)
        except Exception as exception:     
            skip_senders_files.append(file)

In [ ]:
skip_senders_files = get_skip_senders(senders_files['filename'])
tested_senders_files = [file for file in senders_files['filename'] if file not in skip_senders_files]
#len(skip_senders_files)

## encoded values

## init encoders (run only once)

In [21]:
# - run this paragraph only once !
# - if you get an 'allow_pickle' error you need Kernel/restart kernel

import encoder
import numpy as np

np_load_old = np.load

# modify the default parameters of np.load
np.load = lambda *a,**k: np_load_old(*a, allow_pickle=True, **k)

# restore np.load for future normal usage
#np.load = np_load_old

### encode value

In [28]:
def transform(value,column=None, npy= share_directory + 'sla/' + version_sla + '/npy'):
    _encoder = encoder.TolerantLabelEncoder(ignore_unknown=True)
    _encoder.classes_ = np.load(npy + '/' + column + '.npy')
    return int( _encoder.transform([value])[0])

### decode value

In [19]:
def inverse_transform(value,column=None, npy= share_directory + 'sla/' + version_sla + '/npy'):
    _encoder = encoder.TolerantLabelEncoder(ignore_unknown=True)
    _encoder.classes_ = np.load(npy + '/' + column + '.npy')
    if type(value) == int:
        return str(_encoder.inverse_transform(value))  
    elif type(value) == list:
        return [str(_encoder.inverse_transform(v)) for v in value]
    else:
        return None

In [6]:
inverse_transform([6,5],column='CSERVICE')
inverse_transform(6, 'CSTATUS')
#inverse_transform(list(pd.unique(pfall['CSTATUS'])),'CSTATUS')

'ERROR'

## status : was the message processed successfully ?

In [23]:
def check_transformed(pfall,column,value):
    return pfall[column] == transform(value,column=column)

def assign_outcome_error(pfall):
    pfall = pfall.assign(error=(~( ( check_transformed(pfall,'CSTATUS','PENDING') & check_transformed(pfall,'CSERVICE','InvoicePortal')  ) | \
        (check_transformed(pfall,'CSTATUS','PENDING') & check_transformed(pfall,'CSERVICE','IDS')  ) | \
        check_transformed(pfall,'CSTATUS','SUCCESS') | \
        check_transformed(pfall,'CSTATUS','SUCCESS_DOWNLOADED') | check_transformed(pfall,'CSTATUS','SUCCESS_POLLQUEUE')  )).astype(int))
    return pfall

In [ ]:
pfall = assign_outcome_error(pfall)

In [ ]:
pfall[pfall['error'] == 1]

In [ ]:
#pfall = pfall.assign(outcome=(~( ((pfall['CSTATUS'] == 'PENDING') & (pfall['CSERVICE'] == 'InvoicePortal')) | ((pfall['CSTATUS'] == 'PENDING') & (pfall['CSERVICE'] == 'IDS')) | (pfall['CSTATUS'] == 'SUCCESS') | (pfall['CSTATUS'] == 'SUCCESS_DOWNLOADED') | (pfall['CSTATUS'] == 'SUCCESS_POLLQUEUE'))).astype(int))
    